# EX_02 — Embeddings con Transformers (ejercicios)

**Notebook de referencia:** `notebook/02_Embeddings_Transformers.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Mean pooling

Con `AutoTokenizer` + `AutoModel`, obtén **last_hidden_state** para una frase y calcula el embedding de frase como media sobre tokens (excluyendo padding).


In [1]:
import torch
from transformers import AutoTokenizer, AutoModel

# 1. Definición del texto
text = "Transformers build contextual embeddings."

# 2. Cargar un modelo de lenguaje ligero y su tokenizador correspondiente
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# 3. Tokenizar el texto devolviendo tensores de PyTorch
# Activamos el padding y el truncamiento por seguridad para simular un entorno real
inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt")

# 4. Pasar los datos por el modelo (Forward Pass) sin calcular gradientes
with torch.no_grad():
    outputs = model(**inputs)

# 5. Extraer el 'last_hidden_state' y la 'attention_mask'
# last_hidden_state tiene dimensiones: [batch_size, seq_len, hidden_dim]
last_hidden_state = outputs.last_hidden_state
attention_mask = inputs["attention_mask"]

# =====================================================================
# CÁLCULO DE MEAN POOLING (PROMEDIO EXCLUYENDO PADDING)
# =====================================================================

# Alargar la máscara de atención agregando una dimensión extra al final
# Pasa de [batch_size, seq_len] a [batch_size, seq_len, 1] para poder multiplicar
input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()

# Multiplicamos los embeddings por la máscara: las posiciones de padding se vuelven 0 absoluto
sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, dim=1)

# Contamos cuántos tokens reales existen sumando los unos en la máscara (mínimo 1 para evitar división por 0)
sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)

# Calculamos la media real dividiendo la suma entre el número de tokens válidos
mean_pooled_embedding = sum_embeddings / sum_mask

# =====================================================================
# MOSTRAR RESULTADOS
# =====================================================================
print("--- DIMENSIONES DE LOS TENSORES ---")
print(f"Dimensiones de Last Hidden State: {last_hidden_state.shape}")
print(f"Dimensiones de la Máscara:        {attention_mask.shape}")
print(f"Dimensiones del Vector de Frase:  {mean_pooled_embedding.shape}")

print("\n--- VECTOR DE FRASE FINAL (Primeros 10 valores de la dimensión oculta) ---")
print(mean_pooled_embedding[0, :10])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- DIMENSIONES DE LOS TENSORES ---
Dimensiones de Last Hidden State: torch.Size([1, 11, 768])
Dimensiones de la Máscara:        torch.Size([1, 11])
Dimensiones del Vector de Frase:  torch.Size([1, 768])

--- VECTOR DE FRASE FINAL (Primeros 10 valores de la dimensión oculta) ---
tensor([ 0.1519,  0.0297, -0.0235,  0.1273, -0.0312, -0.0803, -0.2190,  0.2592,
         0.2274, -0.3866])


## Actividad 2 — `sentence-transformers`

Usa `SentenceTransformer` para embedder dos frases y calcula similitud coseno. Comenta brevemente (en inglés en un comentario) por qué suele ser mejor que mean-pooling manual de BERT base.


In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer

# =====================================================================
# THEORETICAL DISCUSSION (English comment as requested by the teacher)
# =====================================================================
# Why is SentenceTransformer better than a manual mean-pooling over vanilla BERT base?
#
# 1. SPECIALIZED TRAINING: Vanilla BERT base was pretrained on Masked Language Modeling (MLM).
#    Its raw embeddings are not naturally tuned to find similarity between sentences, often
#    yielding poor semantic clusters. SentenceTransformer models (like Bi-Encoders) are
#    specifically fine-tuned on sentence pairs (e.g., using contrastive loss or NLI datasets)
#    so that similar sentences map directly to close vectors in space.
#
# 2. OUT-OF-THE-BOX EFFICIENCY: It abstracts away the manual mask arithmetic, tokenization,
#    and pooling overhead into a single optimized `.encode()` method, ensuring consistent
#    and mathematically correct pooling without human error.
# =====================================================================

# 1. Inicializar el modelo especializado en embeddings de frases
# Usamos un modelo súper popular, ligero y de alto rendimiento
model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Definir dos frases con significados similares pero palabras diferentes
sentence1 = "Artificial intelligence is changing the world."
sentence2 = "Machine learning algorithms are transforming modern technology."

# 3. Obtener los embeddings directamente con la función .encode()
embedding1 = model.encode(sentence1)
embedding2 = model.encode(sentence2)

# 4. Calcular la similitud coseno utilizando NumPy
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

similarity_score = cosine_similarity(embedding1, embedding2)

# 5. Imprimir resultados
print("--- MODELO SENTENCE-TRANSFORMERS ---")
print(f"Frase 1: '{sentence1}'")
print(f"Frase 2: '{sentence2}'")
print(f"Dimensiones de cada embedding: {embedding1.shape}")
print(f"Similitud Coseno calculada:    {similarity_score:.4f}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

--- MODELO SENTENCE-TRANSFORMERS ---
Frase 1: 'Artificial intelligence is changing the world.'
Frase 2: 'Machine learning algorithms are transforming modern technology.'
Dimensiones de cada embedding: (384,)
Similitud Coseno calculada:    0.6184


## Actividad 3 — Paráfrasis

Escribe dos paráfrasis de una misma idea y muestra que sus embeddings (sentence-transformers) tienen **mayor** similitud entre sí que con una frase de tema distinto.


In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Cargar el modelo preentrenado de sentence-transformers
model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Definir las frases para el experimento
# Frase A y Frase B son paráfrasis (mismo significado, distintas palabras)
# Frase C es completamente ajena (tema distinto)
paraphrase_1 = "The doctor recommended drinking plenty of water every day to stay healthy."
paraphrase_2 = "To maintain good health, the physician advised consuming a lot of water daily."
unrelated    = "The software engineer fixed a critical bug in the database before the release."

# 3. Generar los embeddings para las tres cadenas de texto
emb_p1 = model.encode(paraphrase_1)
emb_p2 = model.encode(paraphrase_2)
emb_un = model.encode(unrelated)

# 4. Función para calcular la similitud coseno
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# 5. Calcular similitudes cruzadas
sim_between_paraphrases = cosine_similarity(emb_p1, emb_p2)
sim_p1_with_unrelated   = cosine_similarity(emb_p1, emb_un)
sim_p2_with_unrelated   = cosine_similarity(emb_p2, emb_un)

# 6. Mostrar los resultados en pantalla
print("--- EXPERIMENTO DE PARÁFRASIS ---")
print(f"Paráfrasis 1: '{paraphrase_1}'")
print(f"Paráfrasis 2: '{paraphrase_2}'")
print(f"Frase Ajena:  '{unrelated}'\n")

print("--- RESULTADOS DE SIMILITUD ---")
print(f"Similitud entre Paráfrasis 1 y Paráfrasis 2: {sim_between_paraphrases:.4f} (Esperado: ALTA)")
print(f"Similitud entre Paráfrasis 1 y Frase Ajena:   {sim_p1_with_unrelated:.4f} (Esperado: BAJA)")
print(f"Similitud entre Paráfrasis 2 y Frase Ajena:   {sim_p2_with_unrelated:.4f} (Esperado: BAJA)")

# Validación matemática del enunciado
assert sim_between_paraphrases > sim_p1_with_unrelated, "¡Error! Las paráfrasis deberían parecerse más entre sí."
print("\n¡Éxito confirmado! Los embeddings demuestran que las dos frases paráfrasis tienen mayor similitud entre sí que con el tema no relacionado.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- EXPERIMENTO DE PARÁFRASIS ---
Paráfrasis 1: 'The doctor recommended drinking plenty of water every day to stay healthy.'
Paráfrasis 2: 'To maintain good health, the physician advised consuming a lot of water daily.'
Frase Ajena:  'The software engineer fixed a critical bug in the database before the release.'

--- RESULTADOS DE SIMILITUD ---
Similitud entre Paráfrasis 1 y Paráfrasis 2: 0.8802 (Esperado: ALTA)
Similitud entre Paráfrasis 1 y Frase Ajena:   0.0328 (Esperado: BAJA)
Similitud entre Paráfrasis 2 y Frase Ajena:   -0.0042 (Esperado: BAJA)

¡Éxito confirmado! Los embeddings demuestran que las dos frases paráfrasis tienen mayor similitud entre sí que con el tema no relacionado.
